In [3]:
# ============================================================
# NOTEBOOK — Adversarial Debiasing: Wilson CI + Statistical
#            Comparison to Group DRO (fixes missing CI claim)
# Dataset: nazmusresan/fitzpatrick17k
# GPU: T4, Internet ON
# Runtime: ~45 min
# After running, paste ALL output back to Claude
# ============================================================

# ── CELL 1: Install + imports ─────────────────────────────────
!pip install torch torchvision transformers scikit-learn \
             pandas numpy imbalanced-learn Pillow -q

import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

warnings.filterwarnings('ignore')

SEEDS        = [42, 0, 1, 7, 99]
RANDOM_STATE = 42
N_BOOTSTRAP  = 1000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Seeds: {SEEDS}")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# ── CELL 2: Wilson CI helper ──────────────────────────────────
def wilson_ci(k, n, z=1.96):
    """Wilson score 95% CI for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return max(0.0, center - margin), min(1.0, center + margin)

def bootstrap_auc(y_true, y_prob, n=N_BOOTSTRAP):
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        try:
            scores.append(roc_auc_score(y_true[idx], y_prob[idx],
                                        multi_class='ovr', average='macro'))
        except: pass
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

# ── CELL 3: Load Fitzpatrick data ─────────────────────────────
print("\nLoading Fitzpatrick17k...")

# ── Path auto-discovery (same pattern as Group DRO notebook) ──
_fitz_csv = None
for _root, _dirs, _files in os.walk('/kaggle/input'):
    for _f in _files:
        if _f.endswith('.csv') and 'fitzpatrick' in _f.lower():
            _fitz_csv = os.path.join(_root, _f)
            break
    if _fitz_csv:
        break

if _fitz_csv:
    FITZ_CSV     = _fitz_csv
    FITZ_IMG_DIR = os.path.dirname(_fitz_csv)   # CSV and images share the same dir
else:
    FITZ_CSV     = '/kaggle/input/fitzpatrick17k/fitzpatrick17k.csv'
    FITZ_IMG_DIR = '/kaggle/input/fitzpatrick17k/images'

print(f"CSV path : {FITZ_CSV}")
print(f"Image dir: {FITZ_IMG_DIR}")
# ──────────────────────────────────────────────────────────────

df = pd.read_csv(FITZ_CSV)
df = df[df['fitzpatrick_scale'] > 0]

# Walk the image dir (handles nested layouts)
image_files = {}
for _img_root, _img_dirs, _img_files in os.walk(FITZ_IMG_DIR):
    for _img_f in _img_files:
        if _img_f.endswith('.jpg') or _img_f.endswith('.png'):
            _key = _img_f.replace('.jpg', '').replace('.png', '')
            image_files[_key] = os.path.join(_img_root, _img_f)
print(f"Image files found: {len(image_files)}")

df['local_path'] = df['md5hash'].map(image_files)
df = df[df['local_path'].notna()].copy()

df['skin_group'] = df['fitzpatrick_scale'].apply(
    lambda x: 'light' if x <= 2 else 'medium' if x <= 4 else 'dark')

light_df  = df[df['skin_group'] == 'light'].sample(1000, random_state=RANDOM_STATE)
medium_df = df[df['skin_group'] == 'medium'].sample(1000, random_state=RANDOM_STATE)
dark_df   = df[df['skin_group'] == 'dark'].sample(
                min(1000, len(df[df['skin_group'] == 'dark'])), random_state=RANDOM_STATE)

print(f"light={len(light_df)}, medium={len(medium_df)}, dark={len(dark_df)}")

le = LabelEncoder()
all_labels_raw = (list(light_df['three_partition_label']) +
                  list(medium_df['three_partition_label']) +
                  list(dark_df['three_partition_label']))
le.fit(all_labels_raw)
print(f"Classes: {le.classes_}")

BENIGN_IDX = list(le.classes_).index('benign')

# ── CELL 4: CLIP feature extraction ──────────────────────────
print("\nLoading CLIP ViT-L/14...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model.eval()

def load_imgs(dataframe):
    imgs, lbls = [], []
    for _, row in dataframe.iterrows():
        try:
            img = Image.open(row['local_path']).convert('RGB').resize((224,224))
            imgs.append(img)
            lbls.append(le.transform([row['three_partition_label']])[0])
        except:
            pass
    return imgs, np.array(lbls)

@torch.no_grad()
def get_features(images, batch_size=32):
    all_feats = []
    for i in range(0, len(images), batch_size):
        batch  = images[i:i+batch_size]
        inputs = clip_proc(images=batch, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        feats  = clip_model.get_image_features(**inputs)
        if not isinstance(feats, torch.Tensor):
            feats = feats.pooler_output if hasattr(feats, 'pooler_output') \
                    else feats.last_hidden_state[:,0]
        feats  = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats)

print("Loading images...")
light_imgs,  light_y  = load_imgs(light_df)
medium_imgs, medium_y = load_imgs(medium_df)
dark_imgs,   dark_y   = load_imgs(dark_df)

print("Extracting features...")
light_feats  = get_features(light_imgs)
medium_feats = get_features(medium_imgs)
dark_feats   = get_features(dark_imgs)
print(f"Shapes: {light_feats.shape}, {medium_feats.shape}, {dark_feats.shape}")

# ── CELL 5: Adversarial Debiasing module ─────────────────────
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class AdvDebiasingProbe(nn.Module):
    def __init__(self, in_dim, n_cls, n_groups, hidden=256, alpha=1.0):
        super().__init__()
        self.alpha     = alpha
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_cls)
        )
        self.discriminator = nn.Sequential(
            nn.Linear(in_dim, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_groups)
        )

    def forward(self, x):
        logits = self.classifier(x)
        rev    = GradientReversalFn.apply(x, self.alpha)
        disc   = self.discriminator(rev)
        return logits, disc

def run_adv_debiasing_seed(seed, train_feats, train_y, train_groups,
                            test_feats, test_y):
    set_seed(seed)
    n_cls    = len(le.classes_)
    n_groups = len(np.unique(train_groups))

    X_tr = torch.FloatTensor(train_feats)
    y_tr = torch.LongTensor(train_y)
    g_tr = torch.LongTensor(train_groups)
    X_te = torch.FloatTensor(test_feats).to(device)

    ds     = TensorDataset(X_tr, y_tr, g_tr)
    loader = DataLoader(ds, batch_size=64, shuffle=True)

    model = AdvDebiasingProbe(train_feats.shape[1], n_cls, n_groups).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    ce    = nn.CrossEntropyLoss()

    for epoch in range(20):
        model.train()
        for xb, yb, gb in loader:
            xb, yb, gb = xb.to(device), yb.to(device), gb.to(device)
            logits, disc = model(xb)
            loss = ce(logits, yb) + 0.5 * ce(disc, gb)
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        logits, _ = model(X_te)
        probs     = torch.softmax(logits, dim=1).cpu().numpy()

    preds = probs.argmax(axis=1)
    try:
        auc = roc_auc_score(test_y, probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')

    # Per-class accuracy
    per_class = {}
    for i, cls in enumerate(le.classes_):
        mask = test_y == i
        if mask.sum() > 0:
            per_class[cls] = float(accuracy_score(test_y[mask], preds[mask]))
        else:
            per_class[cls] = float('nan')

    return {'auc': auc, 'per_class': per_class, 'preds': preds, 'probs': probs}

# ── CELL 6: Main experiment — 5 seeds, demographically-aware split
print("\n" + "="*60)
print("EXPERIMENT: Adversarial Debiasing — 5 seeds, demo-aware split")
print("="*60)

# Build splits (same as rest of codebase)
train_feats = np.vstack([light_feats, medium_feats])
train_y_raw = np.concatenate([light_y, medium_y])

# Dark: 800 test fixed, 200 mitigation pool (matching nb design)
dark_indices = np.arange(len(dark_y))
pool_idx, test_idx = train_test_split(dark_indices, test_size=800,
                                       random_state=RANDOM_STATE,
                                       stratify=dark_y)
dark_test_feats = dark_feats[test_idx]
dark_test_y     = dark_y[test_idx]
dark_pool_feats = dark_feats[pool_idx[:200]]
dark_pool_y     = dark_y[pool_idx[:200]]

# Group labels: 0=light/medium, 1=dark
train_groups = np.zeros(len(train_y_raw), dtype=int)
pool_groups  = np.ones(len(dark_pool_y),  dtype=int)

# Combine training set with dark mitigation pool
full_train_feats  = np.vstack([train_feats, dark_pool_feats])
full_train_y      = np.concatenate([train_y_raw, dark_pool_y])
full_train_groups = np.concatenate([train_groups, pool_groups])

print(f"Train: {len(full_train_y)} samples (pool: {len(dark_pool_y)} dark)")
print(f"Test:  {len(dark_test_y)} dark-skin samples")
print(f"Dark benign in test: {(dark_test_y == BENIGN_IDX).sum()}")

seed_results = []
for seed in SEEDS:
    print(f"\n  Seed {seed}...")
    r = run_adv_debiasing_seed(
        seed, full_train_feats, full_train_y, full_train_groups,
        dark_test_feats, dark_test_y)
    seed_results.append(r)
    print(f"    AUC={r['auc']:.4f}  benign={r['per_class'].get('benign',float('nan')):.4f}")

# ── CELL 7: Aggregate results + Wilson CIs ────────────────────
print("\n" + "="*60)
print("RESULTS: Adversarial Debiasing (5 seeds)")
print("="*60)

aucs          = [r['auc']                         for r in seed_results]
benign_accs   = [r['per_class'].get('benign', 0)  for r in seed_results]
nonneo_accs   = [r['per_class'].get('non-neoplastic', 0) for r in seed_results]
malignant_accs= [r['per_class'].get('malignant', 0) for r in seed_results]

mean_auc    = np.mean(aucs)
std_auc     = np.std(aucs)
mean_benign = np.mean(benign_accs)
std_benign  = np.std(benign_accs)

print(f"\nMean AUC:    {mean_auc:.4f} ± {std_auc:.4f}")
print(f"Mean Benign: {mean_benign:.4f} ± {std_benign:.4f}")
print(f"Mean NonNeo: {np.mean(nonneo_accs):.4f}")
print(f"Mean Malig:  {np.mean(malignant_accs):.4f}")

# Wilson CI on per-seed benign accuracy (aggregate over all test samples)
# Pool all preds from all seeds to get aggregate CI
n_dark_benign = (dark_test_y == BENIGN_IDX).sum()
print(f"\nDark-skin benign test samples: {n_dark_benign}")

# Per-seed Wilson CIs
print("\nPer-seed benign Wilson 95% CIs:")
for i, (seed, r) in enumerate(zip(SEEDS, seed_results)):
    acc = r['per_class'].get('benign', 0)
    k   = round(acc * n_dark_benign)
    lo, hi = wilson_ci(k, n_dark_benign)
    print(f"  Seed {seed}: {acc:.4f} ({lo:.4f}–{hi:.4f})  k={k}/{n_dark_benign}")

# Mean benign acc Wilson CI (using mean k across seeds)
mean_k  = round(mean_benign * n_dark_benign)
lo_mean, hi_mean = wilson_ci(mean_k, n_dark_benign)
print(f"\nMean benign Wilson CI: {mean_benign:.4f} ({lo_mean:.4f}–{hi_mean:.4f})")

# Compare to DRO: do CIs overlap?
# DRO: 0.000% (Wilson CI 0.000–0.019)
dro_hi = 0.019
adv_lo = lo_mean
print(f"\nCI overlap with Group DRO [0.000–0.019]?")
print(f"  Adv debiasing CI lower bound: {adv_lo:.4f}")
print(f"  DRO CI upper bound: {dro_hi:.4f}")
overlap = adv_lo < dro_hi
print(f"  CIs overlap: {overlap}")
print(f"  Statistically distinguishable: {not overlap}")

# ── CELL 8: Baseline comparison (Group DRO reference numbers) ─
print("\n" + "="*60)
print("COMPARISON TABLE: Adv Debiasing vs Group DRO vs Baseline")
print("="*60)

# Reference values from paper (Table 3)
ref = {
    'Baseline':         {'auc': 0.746, 'benign': 0.079, 'benign_ci': (None, None)},
    'Group DRO':        {'auc': 0.701, 'benign': 0.000, 'benign_ci': (0.000, 0.019)},
    'Adv Debiasing':    {'auc': mean_auc, 'benign': mean_benign,
                         'benign_ci': (lo_mean, hi_mean)},
}

print(f"\n{'Intervention':<20} {'Demo AUC':>10} {'Benign Acc':>12} {'95% CI':>20}")
print("-"*65)
for name, v in ref.items():
    ci_str = f"({v['benign_ci'][0]:.3f}–{v['benign_ci'][1]:.3f})" \
             if v['benign_ci'][0] is not None else "from paper"
    print(f"{name:<20} {v['auc']:>10.4f} {v['benign']:>12.4f} {ci_str:>20}")

# ── CELL 9: Confusion matrix (averaged) ──────────────────────
print("\n" + "="*60)
print("CONFUSION MATRIX: Adv Debiasing (seed 42)")
print("="*60)

r0    = seed_results[0]
cm    = confusion_matrix(dark_test_y, r0['preds'], labels=list(range(len(le.classes_))))
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

# ── CELL 10: Save results ─────────────────────────────────────
output = {
    'method': 'adversarial_debiasing',
    'n_seeds': len(SEEDS),
    'seeds': SEEDS,
    'n_dark_benign_test': int(n_dark_benign),
    'mean_auc': float(mean_auc),
    'std_auc': float(std_auc),
    'mean_benign_acc': float(mean_benign),
    'std_benign_acc': float(std_benign),
    'wilson_ci_mean_benign': [float(lo_mean), float(hi_mean)],
    'ci_overlaps_dro': bool(overlap),
    'statistically_distinguishable_from_dro': bool(not overlap),
    'per_seed': [
        {
            'seed': int(SEEDS[i]),
            'auc': float(seed_results[i]['auc']),
            'per_class': {k: float(v) for k, v in seed_results[i]['per_class'].items()}
        }
        for i in range(len(SEEDS))
    ]
}

with open('/kaggle/working/adv_debiasing_ci_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nSaved: /kaggle/working/adv_debiasing_ci_results.json")
print("\n✓ Complete. Paste ALL output back to Claude.")

Device: cuda
Seeds: [42, 0, 1, 7, 99]

Loading Fitzpatrick17k...
CSV path : /kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/fitzpatrick17k (1).csv
Image dir: /kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder
Image files found: 16574
light=1000, medium=1000, dark=1000
Classes: ['benign' 'malignant' 'non-neoplastic']

Loading CLIP ViT-L/14...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading images...
Extracting features...
Shapes: (1000, 768), (1000, 768), (1000, 768)

EXPERIMENT: Adversarial Debiasing — 5 seeds, demo-aware split
Train: 2200 samples (pool: 200 dark)
Test:  800 dark-skin samples
Dark benign in test: 78

  Seed 42...
    AUC=0.7675  benign=0.0256

  Seed 0...
    AUC=0.7696  benign=0.0256

  Seed 1...
    AUC=0.7646  benign=0.0256

  Seed 7...
    AUC=0.7666  benign=0.0256

  Seed 99...
    AUC=0.7667  benign=0.0256

RESULTS: Adversarial Debiasing (5 seeds)

Mean AUC:    0.7670 ± 0.0016
Mean Benign: 0.0256 ± 0.0000
Mean NonNeo: 0.9717
Mean Malig:  0.3482

Dark-skin benign test samples: 78

Per-seed benign Wilson 95% CIs:
  Seed 42: 0.0256 (0.0071–0.0888)  k=2/78
  Seed 0: 0.0256 (0.0071–0.0888)  k=2/78
  Seed 1: 0.0256 (0.0071–0.0888)  k=2/78
  Seed 7: 0.0256 (0.0071–0.0888)  k=2/78
  Seed 99: 0.0256 (0.0071–0.0888)  k=2/78

Mean benign Wilson CI: 0.0256 (0.0071–0.0888)

CI overlap with Group DRO [0.000–0.019]?
  Adv debiasing CI lower bound: 0.0071